<a href="https://colab.research.google.com/github/jstyoon96/WPI-AI-Course/blob/main/WPI_week2/lab1/WPI_week2_lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNN-Based ECG Signal Segmentation

**Audience:** WPI AI Bootcamp students with basic Python experience.

**Estimated time:** 90-120 minutes.

## Learning Objectives

By the end of this lab, you should be able to:

- Convert ECG windows into a segmentation-style dataset with one label per time sample.
- Explain why convolutional filters are useful for local waveform patterns.
- Train a small 1D CNN for R-peak mask prediction.
- Evaluate segmentation behavior with precision, recall, and intersection over union.

## Grading And Word Response Submission

This lab is graded out of **100 pts**.

- Notebook execution and artifacts: **60 pts**
- Word response document: **40 pts**

Use this filename for the Word response document:

`WPI_week2_lab1_responses_LastName_FirstName.docx`

Write Q1-Q5 in the Word document, using 2-5 sentences per question. Keep longer written responses in the Word document rather than in notebook markdown cells.

## Workflow

Run the setup and data cells first. Then work through the model sections in order: inspect the data, build the CNN, train it, evaluate it, and interpret one or two prediction plots.

## Setup

This setup cell installs the small scientific Python stack used in Colab, loads the course helper package, and selects CPU or GPU automatically.

In [ ]:
#@title Setup course environment
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("XDG_CACHE_HOME", "/tmp/.cache")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "scipy",
    "pooch",
    "numpy",
    "pandas",
    "matplotlib",
])

try:
    import torch
except ModuleNotFoundError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch",
        "--index-url",
        "https://download.pytorch.org/whl/cpu",
    ])
    import torch

repo_dir = Path("/content/WPI-AI-Course")
if not repo_dir.parent.exists():
    local_repo = Path.cwd()
    repo_dir = local_repo if (local_repo / "wpi_ai_bootcamp").exists() else Path("/tmp/WPI-AI-Course")

if not repo_dir.exists():
    subprocess.check_call([
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/jstyoon96/WPI-AI-Course.git",
        str(repo_dir),
    ])
elif (repo_dir / ".git").exists() and repo_dir.name == "WPI-AI-Course":
    subprocess.check_call(["git", "-C", str(repo_dir), "pull", "--ff-only"])

sys.path.insert(0, str(repo_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

from wpi_ai_bootcamp.data import make_ecg_mask_dataset
from wpi_ai_bootcamp.notebook import setup_lab

WPI_COLORS = setup_lab()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Setup complete. Device:", DEVICE)

## Data Loading

This lab does not use a committed `.npy` data file. Instead, it builds a compact teaching dataset from the public SciPy ECG sample through `wpi_ai_bootcamp.data.make_ecg_mask_dataset()`.

The mask is derived from a signal-processing peak detector, so treat it as educational supervision rather than clinical ground truth.

In [ ]:
X, y_mask, metadata = make_ecg_mask_dataset(
    window_s=2.0,
    stride_s=0.5,
    mask_radius_s=0.06,
    max_windows=96,
    random_state=7,
)

fs = metadata["fs"]
source = metadata["source"]
time = np.arange(X.shape[-1]) / fs

print("X shape:", X.shape)
print("mask shape:", y_mask.shape)
print("sampling rate:", fs, "Hz")
print("source:", source.name)
print("source url:", source.url)
print("positive mask fraction:", round(float(y_mask.mean()), 4))

source_table = pd.DataFrame([
    {
        "name": source.name,
        "loader": source.loader,
        "citation": source.citation,
        "license note": source.license_note,
    }
])
source_table

## Part 1 — Inspect ECG Windows And Masks

Each input window has shape `(1, samples)`: one ECG channel and a fixed number of time samples. The target mask has one value per time sample, where values near detected R-peaks are labeled `1`.

In [ ]:
def shade_mask_regions(ax, time_values, mask_values, *, label="Target mask", color=None, alpha=0.22):
    """Shade full-height time regions where a binary mask is active."""
    color = color or WPI_COLORS["accent_green"]
    time_values = np.asarray(time_values)
    active = np.asarray(mask_values) >= 0.5
    if not np.any(active):
        return

    edges = np.diff(active.astype(int), prepend=0, append=0)
    starts = np.flatnonzero(edges == 1)
    stops = np.flatnonzero(edges == -1)
    dt = float(np.median(np.diff(time_values))) if len(time_values) > 1 else 0.0
    for region_index, (start, stop) in enumerate(zip(starts, stops)):
        ax.axvspan(
            time_values[start],
            time_values[min(stop, len(time_values) - 1)] + dt,
            color=color,
            alpha=alpha,
            label=label if region_index == 0 else None,
            zorder=0,
        )

# TODO: Change EXAMPLE_INDEX and describe how the mask aligns with the ECG morphology.
EXAMPLE_INDEX = 0

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(time, X[EXAMPLE_INDEX, 0], color=WPI_COLORS["black"], linewidth=1.0, label="ECG window")
shade_mask_regions(ax, time, y_mask[EXAMPLE_INDEX], label="R-peak mask", color=WPI_COLORS["crimson"], alpha=0.20)
ax.set(title=f"ECG window {EXAMPLE_INDEX}", xlabel="Time within window (s)", ylabel="Normalized amplitude")
ax.legend(loc="upper right")
plt.show()

### Part 1 Assessment — Data Inspection (20 pts)

- Notebook artifact: a plotted ECG window with the target mask overlaid. **12 pts**
- Word response Q1: Where does the mask align well with the waveform, and where might it be ambiguous? **8 pts**

TODO: In your response document, explain why a segmentation mask is a different target than a single class label.

## Part 2 — Train/Validation/Test Split

We split by window so the model can learn on one subset and be evaluated on held-out windows. The windows overlap, so the reported metrics are for learning behavior in this lab, not a clinical performance claim.

In [ ]:
class ECGMaskDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

rng = np.random.default_rng(11)
indices = rng.permutation(len(X))
n_train = int(0.70 * len(indices))
n_val = int(0.15 * len(indices))
train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

dataset = ECGMaskDataset(X, y_mask)

# TODO: Try a different batch size after you get the baseline run working.
BATCH_SIZE = 16

train_loader = DataLoader(Subset(dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(Subset(dataset, val_idx), batch_size=BATCH_SIZE)
test_loader = DataLoader(Subset(dataset, test_idx), batch_size=BATCH_SIZE)

print("train/val/test windows:", len(train_idx), len(val_idx), len(test_idx))

## Part 3 — Build A 1D CNN Segmenter

A 1D CNN slides small filters over the signal. Early layers can learn local edges and QRS-like shapes, while later layers combine those patterns into a per-sample mask prediction.

In [ ]:
class CNNMaskSegmenter(nn.Module):
    def __init__(self, hidden_channels=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, hidden_channels, kernel_size=9, padding=4),
            nn.BatchNorm1d(hidden_channels),
            nn.ReLU(),
            nn.Conv1d(hidden_channels, hidden_channels, kernel_size=9, padding=4),
            nn.BatchNorm1d(hidden_channels),
            nn.ReLU(),
            nn.Conv1d(hidden_channels, hidden_channels // 2, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(hidden_channels // 2, 1, kernel_size=1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

# TODO: Try 16 or 48 hidden channels after the first complete run.
cnn_model = CNNMaskSegmenter(hidden_channels=32).to(DEVICE)

total_params = sum(p.numel() for p in cnn_model.parameters())
print(cnn_model)
print("trainable parameters:", total_params)

## Part 4 — Train The CNN

R-peak masks are sparse: most time samples are background. The positive class weight below helps the loss pay attention to the smaller mask region.

In [ ]:
def mask_metrics(logits, targets, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = probs >= threshold
    truth = targets >= 0.5
    tp = (preds & truth).sum().item()
    fp = (preds & ~truth).sum().item()
    fn = (~preds & truth).sum().item()
    union = (preds | truth).sum().item()
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    iou = tp / (union + 1e-8)
    return {"precision": precision, "recall": recall, "iou": iou}


def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
    return total_loss / len(loader.dataset)


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    collected = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            total_loss += loss.item() * len(xb)
            collected.append(mask_metrics(logits.cpu(), yb.cpu()))
    metrics = pd.DataFrame(collected).mean().to_dict()
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics

In [ ]:
positive_fraction = float(y_mask[train_idx].mean())
pos_weight = torch.tensor([(1.0 - positive_fraction) / max(positive_fraction, 1e-6)], device=DEVICE)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-3)

# TODO: Increase EPOCHS if your validation loss is still clearly improving.
EPOCHS = 4
history = []
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(cnn_model, train_loader, optimizer, loss_fn)
    val_metrics = evaluate(cnn_model, val_loader, loss_fn)
    row = {"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(row)
    print(row)

pd.DataFrame(history)

### Part 4 Assessment — Training Behavior (25 pts)

- Notebook artifact: the training history table from the CNN run. **15 pts**
- Word response Q2: Did the validation metrics improve smoothly, flatten out, or move inconsistently? What might explain that behavior? **10 pts**

## Part 5 — Evaluate And Visualize Predictions

Precision, recall, and IoU summarize different failure modes. The plot helps you connect those metrics back to the ECG waveform.

In [ ]:
test_metrics = evaluate(cnn_model, test_loader, loss_fn)
pd.DataFrame([test_metrics])

In [ ]:
def plot_prediction(model, dataset, sample_position=0, title="Model prediction"):
    model.eval()
    xb, yb = dataset[sample_position]
    with torch.no_grad():
        logits = model(xb.unsqueeze(0).to(DEVICE)).cpu().squeeze(0)
    probs = torch.sigmoid(logits).numpy()

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(time, xb.squeeze(0).numpy(), color=WPI_COLORS["black"], linewidth=1.0, label="ECG window")
    shade_mask_regions(ax, time, yb.numpy(), label="Target mask", color=WPI_COLORS["accent_green"], alpha=0.20)
    ax.plot(time, probs, color=WPI_COLORS["crimson"], linewidth=1.5, label="Predicted probability")
    ax.set(title=title, xlabel="Time within window (s)", ylabel="Normalized signal / probability")
    ax.legend(loc="upper right")
    plt.show()

# TODO: Change SAMPLE_POSITION to inspect several test windows.
SAMPLE_POSITION = 0
plot_prediction(cnn_model, Subset(dataset, test_idx), SAMPLE_POSITION, "CNN mask prediction")

### Part 5 Assessment — Error Analysis (35 pts)

- Notebook artifact: a test metric table and at least one prediction plot. **20 pts**
- Word response Q3: Identify one false positive or false negative region, or explain why the selected example looks accurate. **8 pts**
- Word response Q4: Which metric would you prioritize if missing an R-peak is worse than drawing a slightly wide mask, and why? **7 pts**

TODO: Include the sample position you inspected in your response.

## Optional Challenge

TODO: Change `mask_radius_s` in the data-loading cell and rerun the notebook. How does a wider or narrower target mask change the apparent difficulty of the segmentation task?

## Final Reflection

Word response Q5: In 3-5 sentences, explain one strength and one limitation of using a CNN for ECG mask prediction in this lab.

## Attribution

- Data: SciPy electrocardiogram sample, loaded through `scipy.datasets.electrocardiogram()` via `wpi_ai_bootcamp.data.make_ecg_mask_dataset()`.
- Source note: SciPy documents the sample as derived from MIT-BIH Arrhythmia Database record 208 on PhysioNet.
- Libraries: NumPy, pandas, Matplotlib, SciPy, and PyTorch.
- WPI colors: WPI Institutional Guidelines, Crimson `#AC2B37` and Gray `#A9B0B7`.